# Practice 60 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Student exam scores

`scores` records `math` and `science` exam results for six students across two semesters.

1. Reshape into long format — one row per `(student_id, cohort, semester)`. No syntax reminder.
2. Did scores improve from S1 to S2? For each subject, compute the mean score per semester using `np.mean`, and report the difference.
3. Use `np.corrcoef` to check whether `math` and `science` track together across all rows.

In [3]:
scores = pd.DataFrame({
    'student_id': [1, 2, 3, 4, 5, 6],
    'cohort':     ['X', 'Y', 'X', 'Y', 'X', 'Y'],
    'math_S1':    [88, 72, 95, 68, 83, 77],
    'math_S2':    [91, 78, 93, 74, 89, 80],
    'science_S1': [75, 65, 88, 70, 79, 66],
    'science_S2': [80, 70, 90, 73, 82, 71],
})

# Your code here


scores_long = pd.wide_to_long(
    scores, 
    stubnames=['math','science'],
    i = ['student_id','cohort'],
    j = 'semester',
    sep = '_',
    suffix = r'\w+'
)

In [14]:
math_m = scores_long.groupby(level='semester')['math'].mean()
science_m = scores_long.groupby(level='semester')['science'].mean()
print(math_m)
print(science_m)
print('math semester diff: ',math_m['S2'] - math_m['S1'])
print ('science semester diff: ',science_m['S2'] - science_m['S1'])
cor = np.corrcoef(scores_long['math'], scores_long['science'])[0,1]
print('correlation is ', cor, 'highly correlated')

semester
S1    80.500000
S2    84.166667
Name: math, dtype: float64
semester
S1    73.833333
S2    77.666667
Name: science, dtype: float64
math semester diff:  3.6666666666666714
science semester diff:  3.833333333333343
correlation is  0.8690218576766615 highly correlated


---

## Level 2 — Diamonds

Use `sns.load_dataset('diamonds')`. Investigate how cut quality and color grade relate.

1. Build a crosstab of `cut` vs `color`, `margins=True`. In absolute terms, which color grade has the most `Ideal`-cut diamonds? Exclude the `'All'` row/column when needed.
2. Normalize by column. Among `D`-color diamonds, which cut is most common?
3. Add `value_score = price / carat ** 2`. For each `cut`, use `np.percentile` to get the 25th and 75th percentiles of `value_score`. Use `np.argsort` to rank cuts from smallest to largest IQR — which cut has the tightest value spread?

In [57]:
diamonds = sns.load_dataset('diamonds')

# Your code here
cp =pd.crosstab(diamonds['cut'], diamonds['color'], margins=True)
print(cp)
print(cp.loc['Ideal',[col for col in cp.columns if col != 'All']].idxmax())
cpn = pd.crosstab(diamonds['cut'], diamonds['color'], normalize='columns')
print(cpn)
print(cpn['D'].idxmax())
diamonds['value_score'] = diamonds['price']/(diamonds['carat']**2)
dg = diamonds.groupby('cut', observed=True)['value_score'].apply(lambda x: np.percentile(x, [25, 75]))
print(dg)
#iqr = [dg[i][1] - dg[i][0] for i in dg.index]
iqr1 = dg.apply(lambda x: x[1] - x[0])
print(iqr1.iloc[np.argsort(iqr1)])
print(iqr1.iloc[np.argsort(iqr1)].index[0], ' has the tightest IQR')

color         D     E     F      G     H     I     J    All
cut                                                        
Ideal      2834  3903  3826   4884  3115  2093   896  21551
Premium    1603  2337  2331   2924  2360  1428   808  13791
Very Good  1513  2400  2164   2299  1824  1204   678  12082
Good        662   933   909    871   702   522   307   4906
Fair        163   224   312    314   303   175   119   1610
All        6775  9797  9542  11292  8304  5422  2808  53940
G
color             D         E         F         G         H         I  \
cut                                                                     
Ideal      0.418303  0.398387  0.400964  0.432519  0.375120  0.386020   
Premium    0.236605  0.238542  0.244288  0.258944  0.284200  0.263371   
Very Good  0.223321  0.244973  0.226787  0.203595  0.219653  0.222058   
Good       0.097712  0.095233  0.095263  0.077134  0.084538  0.096274   
Fair       0.024059  0.022864  0.032698  0.027807  0.036488  0.032276   

color 

---

## Level 3 — Titanic

Use `sns.load_dataset('titanic')`. No sub-steps — you decide the approach.

1. Which embarkation town (`embark_town`) sent the highest share of first-class passengers? Use a crosstab normalized by row (exclude rows where `embark_town` is null).
2. Use `np.corrcoef` to compare how `fare` and `age` each correlate with `survived`. Drop rows where either is null before computing. Which matters more?
3. Filter to women only. Use `np.percentile` to split at the median fare. Label each woman `'high'` or `'low'`. What is the survival rate for each group?

In [68]:
titanic = sns.load_dataset('titanic')

# Your code here
pt = pd.crosstab(titanic['embark_town'], titanic['pclass'], normalize='index')
print(pt)
print(pt[1].idxmax(),'has the highst share of first-class passengers')

titanic2 = titanic.dropna(subset=['fare', 'age'])
fc = np.corrcoef(titanic2['fare'], titanic2['survived'])[0,1]
ac = np.corrcoef(titanic2['age'], titanic2['survived'])[0,1]
print(fc, ac )
print('fare matters more')

ft = titanic.loc[titanic['sex'] =='female'].copy()
ft['fare_class'] = np.where(ft['fare']>np.percentile(ft['fare'],50), 'high','low')
ft.groupby('fare_class')['survived'].mean()

pclass              1         2         3
embark_town                              
Cherbourg    0.505952  0.101190  0.392857
Queenstown   0.025974  0.038961  0.935065
Southampton  0.197205  0.254658  0.548137
Cherbourg has the highst share of first-class passengers
0.2681886168744788 -0.07722109457217768
fare matters more


fare_class
high    0.814103
low     0.670886
Name: survived, dtype: float64